In [1]:
# @title
# ===== Colab only: fetch the project files from GitHub — run FIRST =====
# Opening a notebook from GitHub loads ONLY the notebook; the repository
# files (source/, preprocessed/, results/, ...) are not there. This cell
# clones the repo into the runtime and moves into it.
#
# For a private repo use
#   https://<TOKEN>@github.com/<user>/<repo>.git
# with a GitHub personal access token.
#
# Run this cell again after every session restart: the clone survives on
# disk, but the working directory resets — the cell then only cd's back.
# Skip this cell entirely when running the notebook locally.
REPO_URL = "https://github.com/karkakol/TranslatorData.git"

from pathlib import Path

repo_name = REPO_URL.rstrip("/").removesuffix(".git").rsplit("/", 1)[-1]
if Path.cwd().name == repo_name:
    print(f"Already inside {Path.cwd()}")
else:
    if not Path(repo_name).is_dir():
        !git clone {REPO_URL}
    %cd {repo_name}


/content/TranslatorData


In [2]:
# @title
# ===== Setup for COMET / BERTScore (safe to re-run) =====
# Installs only what is missing, so re-running this cell is harmless.
# After a run that actually installed something you MUST restart the
# session (Colab: Runtime -> Restart session) and then re-run:
# clone cell (Colab), load cell, definitions cell, runner cell.
#
# The forced protobuf pin fixes the "cannot import name 'runtime_version'"
# error on Colab: COMET's stack needs protobuf 5.x, but Colab preinstalls
# 4.x and pip's resolver refuses to upgrade it (TensorFlow pins <5).
# The pip warning that tensorflow is "incompatible" is expected and harmless.
import importlib.metadata
import importlib.util


def protobuf_needs_pin():
    try:
        return int(importlib.metadata.version("protobuf").split(".")[0]) < 5
    except importlib.metadata.PackageNotFoundError:
        return False


need_packages = not (
    importlib.util.find_spec("comet") and importlib.util.find_spec("bert_score")
)
if need_packages:
    %pip install unbabel-comet bert-score

need_protobuf = protobuf_needs_pin()
if need_protobuf:
    %pip install --no-deps --force-reinstall "protobuf==5.29.3"

if need_packages or need_protobuf:
    print("\n*** Installed something new -> RESTART the session now, ***")
    print("*** then re-run: clone (Colab), load, definitions, runner. ***")
else:
    print("Everything already installed — nothing to do.")


Everything already installed — nothing to do.


In [3]:
# @title
import csv
from pathlib import Path

# Preprocessing: source/en-pl-pairs.tsv -> preprocessed/*.csv (app inputs).
INPUT = "source/en-pl-pairs.tsv"
OUTPUT_DIR = Path("preprocessed")
OUTPUT_DIR.mkdir(exist_ok=True)

# The TSV has 4 columns: pl_id, pl_text, en_id, en_text.
# It also starts with a UTF-8 BOM, so we read with utf-8-sig to strip it.
pairs = []  # list of (pl, en)
with open(INPUT, encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f, delimiter="\t")
    for row in reader:
        if len(row) < 4:
            continue  # skip malformed / empty lines
        pl, en = row[1].strip(), row[3].strip()
        if pl and en:
            pairs.append((pl, en))

print(f"Parsed {len(pairs)} pairs")

# The apps split each line on ';' (source;gold), so sentences containing a
# semicolon would break parsing. Drop those pairs (~0.13% of the data).
before = len(pairs)
pairs = [(pl, en) for pl, en in pairs if ";" not in pl and ";" not in en]
print(f"Dropped {before - len(pairs)} pairs containing semicolons")


def dedup_pairs(pairs, key):
    """Keep the first pair for each distinct key(pair), preserving order."""
    seen = set()
    out = []
    for pair in pairs:
        k = key(pair)
        if k not in seen:
            seen.add(k)
            out.append(pair)
    return out


# One gold translation per distinct source sentence (first occurrence wins).
pl_en = dedup_pairs(pairs, key=lambda p: p[0])  # pl -> en direction
en_pl = dedup_pairs(pairs, key=lambda p: p[1])  # en -> pl direction
print(f"pl->en: {len(pl_en)} unique sources, en->pl: {len(en_pl)} unique sources")


def write_pairs(name, header, rows):
    """Benchmark input file: a header line, then `source;gold` lines.
    Plain text (no CSV quoting) so the apps can simply split on ';'."""
    path = OUTPUT_DIR / name
    with open(path, "w", encoding="utf-8") as f:
        f.write(header + "\n")
        f.writelines(f"{source};{gold}\n" for source, gold in rows)
    print(f"Wrote {path} ({len(rows)} rows)")


write_pairs("pl.csv", "pl;en", pl_en)
write_pairs("en.csv", "en;pl", [(en, pl) for pl, en in en_pl])
write_pairs("pl_short.csv", "pl;en", pl_en[:100])
write_pairs("en_short.csv", "en;pl", [(en, pl) for pl, en in en_pl[:100]])


Parsed 83980 pairs
Dropped 111 pairs containing semicolons
pl->en: 78296 unique sources, en->pl: 75390 unique sources
Wrote preprocessed/pl.csv (78296 rows)
Wrote preprocessed/en.csv (75390 rows)
Wrote preprocessed/pl_short.csv (100 rows)
Wrote preprocessed/en_short.csv (100 rows)


# Evaluation

The cells below score the benchmark results produced by the iOS and Android apps.
See **[EVALUATION.md](EVALUATION.md)** for the full explanation: file naming, what
each metric measures, the single-reference caveat, and how to read the
significance test.

Session order (the setup cells live at the very top of the notebook):

1. **Clone cell** (top of notebook, Colab only) — fetches the repo files.
2. **Setup cell** (top of notebook, run once) — installs COMET/BERTScore
   packages; restart the session afterwards.
3. **Load cell** below — discovers all `ios_*` / `android_*` files in `results/`.
4. **Definitions cell** — pure-stdlib chrF and BLEU plus the `evaluate(size)`
   function (score table → paired bootstrap → largest disagreements).
5. **Cross-check cell** (optional) — validates chrF/BLEU against sacrebleu.
6. **Runner cell** — `comet_evaluate`, `bertscore_evaluate`, and `run_all(size)`,
   which runs all four metrics in one call. GPU is detected automatically.
7. **`run_all("short")`** — everything on the 100-sentence sets (smoke test).
8. **`run_all("full")`** — the thesis run; use a Colab T4 GPU runtime.
9. **Download cell** — zips `evaluation/` and downloads it (on Colab).

All metrics write into `evaluation/scores.csv` (each fills only its own columns,
so partial runs merge), and `evaluate()` also writes
`evaluation/disagreements_{direction}_{size}.csv`. Without the setup packages,
COMET/BERTScore skip with a message, so `run_all` still works locally.


In [4]:
# @title
from pathlib import Path

# Load benchmark result files copied from the devices into results/.
# Expected names: {platform}_{src}-{tgt}_{size}.csv, e.g. ios_pl-en_short.csv,
# each with a `text;translation;gold` header.
# Standard library only — no pandas needed.


def parse_line(line):
    """text and gold never contain ';' (preprocessing dropped such pairs),
    but the model's own translation may — rejoin the middle parts."""
    parts = line.split(";")
    if len(parts) < 3:
        raise ValueError(f"malformed line: {line!r}")
    return parts[0], ";".join(parts[1:-1]), parts[-1]


results = {}  # (platform, direction, size) -> {"text": [...], "translation": [...], "gold": [...]}
for path in sorted(Path("results").glob("*.csv")):
    parts = path.stem.split("_")
    if len(parts) != 3 or "-" not in parts[1]:
        print(f"Skipping {path.name}: name doesn't match platform_direction_size")
        continue
    platform, direction, size = parts
    lines = path.read_text(encoding="utf-8").splitlines()
    assert lines[0] == "text;translation;gold", f"unexpected header in {path}"
    rows = [parse_line(l) for l in lines[1:] if l.strip()]
    results[(platform, direction, size)] = {
        "text": [r[0] for r in rows],
        "translation": [r[1] for r in rows],
        "gold": [r[2] for r in rows],
    }
    print(f"Loaded {path.name}: {len(rows)} rows")

if not results:
    print("No result files found — copy the ios_* / android_* CSVs into results/ first.")


Loaded android_en-pl_full.csv: 75390 rows
Loaded android_en-pl_short.csv: 100 rows
Loaded android_pl-en_full.csv: 78296 rows
Loaded android_pl-en_short.csv: 100 rows
Loaded ios_en-pl_full.csv: 75390 rows
Loaded ios_en-pl_short.csv: 100 rows
Loaded ios_pl-en_full.csv: 78296 rows
Loaded ios_pl-en_short.csv: 100 rows


In [5]:
# @title
import csv
import math
import random
import re
from collections import Counter
from pathlib import Path

# Pure-Python chrF and BLEU — standard library only.
#
# chrF (Popović 2015): character n-grams 1..6, beta=2 (recall weighted 2x),
# whitespace removed, per-order F-scores averaged — the same defaults
# sacrebleu uses. Higher is better, range 0-100.
#
# BLEU (Papineni et al. 2002): word 1..4-gram precision with a brevity
# penalty. The tokenizer splits words and punctuation, close to
# sacrebleu's "13a". Higher is better, range 0-100.

MAX_ORDER = 6
BETA = 2.0


def char_ngrams(text, n):
    chars = re.sub(r"\s+", "", text)
    return Counter(chars[i : i + n] for i in range(len(chars) - n + 1))


def chrf_stats(hypothesis, reference):
    """Per-sentence chrF statistics: (matched, hyp_total, ref_total) per
    n-gram order. Precomputing these makes corpus scores and bootstrap
    resampling cheap sums instead of repeated n-gram counting."""
    stats = []
    for n in range(1, MAX_ORDER + 1):
        hyp_grams = char_ngrams(hypothesis, n)
        ref_grams = char_ngrams(reference, n)
        stats.append((
            sum((hyp_grams & ref_grams).values()),
            sum(hyp_grams.values()),
            sum(ref_grams.values()),
        ))
    return stats


def chrf_from_stats(stats_list, indices=None):
    """Corpus chrF from precomputed per-sentence statistics."""
    sums = [[0, 0, 0] for _ in range(MAX_ORDER)]
    for i in indices if indices is not None else range(len(stats_list)):
        for n, (matched, hyp_total, ref_total) in enumerate(stats_list[i]):
            sums[n][0] += matched
            sums[n][1] += hyp_total
            sums[n][2] += ref_total
    f_scores = []
    for matched, hyp_total, ref_total in sums:
        precision = matched / hyp_total if hyp_total else 0.0
        recall = matched / ref_total if ref_total else 0.0
        if precision + recall > 0:
            f = (1 + BETA**2) * precision * recall / (BETA**2 * precision + recall)
        else:
            f = 0.0
        f_scores.append(f)
    return 100 * sum(f_scores) / MAX_ORDER


def chrf(hypotheses, references):
    return chrf_from_stats([chrf_stats(h, r) for h, r in zip(hypotheses, references)])


TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def tokenize(text):
    return TOKEN_RE.findall(text)


def word_ngrams(tokens, n):
    return Counter(tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1))


def bleu(hypotheses, references, max_n=4):
    """Corpus BLEU with brevity penalty (no smoothing)."""
    matched, totals = [0] * max_n, [0] * max_n
    hyp_len = ref_len = 0
    for hyp, ref in zip(hypotheses, references):
        hyp_tokens, ref_tokens = tokenize(hyp), tokenize(ref)
        hyp_len += len(hyp_tokens)
        ref_len += len(ref_tokens)
        for n in range(1, max_n + 1):
            hyp_grams = word_ngrams(hyp_tokens, n)
            ref_grams = word_ngrams(ref_tokens, n)
            matched[n - 1] += sum((hyp_grams & ref_grams).values())
            totals[n - 1] += sum(hyp_grams.values())
    precisions = [m / t if t else 0.0 for m, t in zip(matched, totals)]
    if min(precisions) == 0.0 or hyp_len == 0:
        return 0.0
    log_avg = sum(math.log(p) for p in precisions) / max_n
    brevity = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / hyp_len)
    return 100 * brevity * math.exp(log_avg)


def paired_data(results_a, results_b):
    """Join two result sets on the source text. Android writes rows in
    completion order, so row-by-row comparison would be wrong."""
    b_map = {
        text: translation
        for text, translation in zip(results_b["text"], results_b["translation"])
    }
    texts, hyp_a, hyp_b, gold = [], [], [], []
    for text, translation, g in zip(
        results_a["text"], results_a["translation"], results_a["gold"]
    ):
        if text in b_map:
            texts.append(text)
            hyp_a.append(translation)
            hyp_b.append(b_map[text])
            gold.append(g)
    return texts, hyp_a, hyp_b, gold


# ----- persistent outputs -----
# evaluation/scores.csv accumulates every metric per (platform, direction,
# size). Each metric writer fills only its own columns, so chrF/BLEU (local)
# and COMET/BERTScore (Colab) can be added independently at different times.
SCORES_PATH = Path("evaluation") / "scores.csv"
SCORE_FIELDS = [
    "platform", "direction", "size", "sentences",
    "chrF", "BLEU", "bootstrap_win", "COMET", "BERTScore",
]


def save_scores(updates):
    """Merge {(platform, direction, size): {column: value}} into scores.csv."""
    SCORES_PATH.parent.mkdir(exist_ok=True)
    table = {}
    if SCORES_PATH.exists():
        with open(SCORES_PATH, encoding="utf-8", newline="") as f:
            for row in csv.DictReader(f, delimiter=";"):
                table[(row["platform"], row["direction"], row["size"])] = row
    for key, values in updates.items():
        row = table.setdefault(key, dict(zip(["platform", "direction", "size"], key)))
        row.update({column: str(value) for column, value in values.items()})
    with open(SCORES_PATH, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=SCORE_FIELDS, delimiter=";", restval="")
        writer.writeheader()
        for key in sorted(table):
            writer.writerow({field: table[key].get(field, "") for field in SCORE_FIELDS})
    print(f"Updated {SCORES_PATH}")


def evaluate(size, n_bootstrap=1000, bootstrap_cap=10_000, top_n=10, seed=42):
    """The complete evaluation flow for one data set size:
    1. corpus chrF/BLEU per platform and direction,
    2. paired bootstrap significance test on chrF,
    3. the sentences with the largest per-sentence chrF gap.
    Saves scores into evaluation/scores.csv and the disagreements into
    evaluation/disagreements_{direction}_{size}.csv.
    For the full set the bootstrap resamples at most `bootstrap_cap`
    sentences per iteration to keep the runtime reasonable."""
    subset = {key: value for key, value in results.items() if key[2] == size}
    if not subset:
        print(f"No '{size}' result files loaded.")
        return

    SCORES_PATH.parent.mkdir(exist_ok=True)

    updates = {}
    print(f"===== {size.upper()} DATA =====\n")
    print(f"{'platform':<9} {'direction':<10} {'sent':>6} {'chrF':>7} {'BLEU':>7}")
    for key in sorted(subset, key=lambda k: (k[1], k[0])):
        platform, direction, _ = key
        df = subset[key]
        chrf_score = chrf(df["translation"], df["gold"])
        bleu_score = bleu(df["translation"], df["gold"])
        updates[key] = {
            "sentences": len(df["text"]),
            "chrF": f"{chrf_score:.2f}",
            "BLEU": f"{bleu_score:.2f}",
        }
        print(
            f"{platform:<9} {direction:<10} {len(df['text']):>6} "
            f"{chrf_score:>7.2f} {bleu_score:>7.2f}"
        )

    for direction in sorted({key[1] for key in subset}):
        ios = results.get(("ios", direction, size))
        android = results.get(("android", direction, size))
        if ios is None or android is None:
            continue
        texts, hyp_ios, hyp_android, gold = paired_data(ios, android)
        stats_ios = [chrf_stats(h, g) for h, g in zip(hyp_ios, gold)]
        stats_android = [chrf_stats(h, g) for h, g in zip(hyp_android, gold)]

        # 2. Paired bootstrap on chrF.
        rng = random.Random(seed)
        n = len(gold)
        sample_size = min(n, bootstrap_cap)
        wins_ios = wins_android = 0
        for _ in range(n_bootstrap):
            idx = [rng.randrange(n) for _ in range(sample_size)]
            score_ios = chrf_from_stats(stats_ios, idx)
            score_android = chrf_from_stats(stats_android, idx)
            if score_ios > score_android:
                wins_ios += 1
            elif score_android > score_ios:
                wins_android += 1
        verdict = "SIGNIFICANT" if max(wins_ios, wins_android) >= 0.95 * n_bootstrap else "tie"
        print(
            f"\n{direction} paired bootstrap (n={n}, sample={sample_size}, "
            f"{n_bootstrap} iterations): iOS wins {wins_ios / n_bootstrap:.1%}, "
            f"Android wins {wins_android / n_bootstrap:.1%} -> {verdict}"
        )
        for platform, wins in (("ios", wins_ios), ("android", wins_android)):
            if (platform, direction, size) in updates:
                updates[(platform, direction, size)]["bootstrap_win"] = (
                    f"{wins / n_bootstrap:.3f}"
                )

        # 3. Largest per-sentence disagreements.
        rows = sorted(
            (
                (
                    abs(chrf_from_stats([a]) - chrf_from_stats([b])),
                    text,
                    hyp_i,
                    chrf_from_stats([a]),
                    hyp_a,
                    chrf_from_stats([b]),
                    g,
                )
                for text, hyp_i, hyp_a, g, a, b in zip(
                    texts, hyp_ios, hyp_android, gold, stats_ios, stats_android
                )
            ),
            reverse=True,
        )
        disagreements_path = SCORES_PATH.parent / f"disagreements_{direction}_{size}.csv"
        with open(disagreements_path, "w", encoding="utf-8", newline="") as f:
            writer = csv.writer(f, delimiter=";")
            writer.writerow(["gap", "text", "ios", "ios_chrf", "android", "android_chrf", "gold"])
            for gap, text, hyp_i, score_i, hyp_a, score_a, g in rows[:top_n]:
                writer.writerow([f"{gap:.1f}", text, hyp_i, f"{score_i:.1f}", hyp_a, f"{score_a:.1f}", g])
        print(f"Wrote {disagreements_path}")

        print(f"\n{direction}: {top_n} largest disagreements")
        for _, text, hyp_i, score_i, hyp_a, score_a, g in rows[:top_n]:
            print(f"src:              {text}")
            print(f"ios ({score_i:5.1f}):      {hyp_i}")
            print(f"android ({score_a:5.1f}):  {hyp_a}")
            print(f"gold:             {g}")
            print()

    save_scores(updates)


In [6]:
# @title
# Optional cross-check: compare the pure-Python chrF/BLEU with sacrebleu
# (the reference implementation used in MT literature).
# Small BLEU differences are expected from tokenization details.
# Runs on the short sets only — validating the implementation needs 100
# sentences, not 78k, and a GPU does not speed up these string metrics.
CROSSCHECK_SIZE = "short"

try:
    import sacrebleu
except ImportError:
    print("sacrebleu not installed — skip, or run: pip install sacrebleu")
else:
    for key, df in sorted(results.items()):
        if key[2] != CROSSCHECK_SIZE:
            continue
        refs = [df["gold"]]
        sb_chrf = sacrebleu.corpus_chrf(df["translation"], refs).score
        sb_bleu = sacrebleu.corpus_bleu(df["translation"], refs).score
        own_chrf = chrf(df["translation"], df["gold"])
        own_bleu = bleu(df["translation"], df["gold"])
        print(
            f"{key}: chrF here={own_chrf:.2f} sacrebleu={sb_chrf:.2f} | "
            f"BLEU here={own_bleu:.2f} sacrebleu={sb_bleu:.2f}"
        )


('android', 'en-pl', 'short'): chrF here=60.88 sacrebleu=60.88 | BLEU here=34.73 sacrebleu=34.71
('android', 'pl-en', 'short'): chrF here=62.62 sacrebleu=62.62 | BLEU here=42.50 sacrebleu=44.26
('ios', 'en-pl', 'short'): chrF here=70.68 sacrebleu=70.68 | BLEU here=46.90 sacrebleu=46.87
('ios', 'pl-en', 'short'): chrF here=74.36 sacrebleu=74.36 | BLEU here=59.77 sacrebleu=58.73


In [7]:
# @title
# ===== Neural metrics + combined runner =====
# comet_evaluate(size) and bertscore_evaluate(size) score one data-set
# size and save into evaluation/scores.csv. Both need the setup cell
# above (install + session restart); without the packages they skip with
# a message instead of failing. A GPU is used automatically when the
# runtime has one.
#
# run_all(size) runs ALL FOUR metrics: chrF/BLEU (with bootstrap and
# disagreement files) via evaluate(), then COMET, then BERTScore.


def comet_evaluate(size):
    try:
        import torch
        from comet import download_model, load_from_checkpoint
    except ImportError as error:
        print(f"COMET unavailable — skipping: {error!r}")
        print("Run the setup cell and RESTART the session to enable it.")
        return
    gpus = 1 if torch.cuda.is_available() else 0
    print(f"COMET on {'GPU' if gpus else 'CPU'}")
    checkpoint = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))
    updates = {}
    for (platform, direction, this_size), df in sorted(results.items()):
        if this_size != size:
            continue
        data = [
            {"src": src, "mt": mt, "ref": ref}
            for src, mt, ref in zip(df["text"], df["translation"], df["gold"])
        ]
        score = checkpoint.predict(data, batch_size=32, gpus=gpus).system_score
        print(f"{platform} {direction} {this_size}: COMET = {score:.4f}")
        updates[(platform, direction, this_size)] = {
            "COMET": f"{score:.4f}",
            "sentences": len(df["text"]),
        }
    if updates:
        save_scores(updates)


def bertscore_evaluate(size):
    try:
        from bert_score import score as bert_score
    except ImportError as error:
        print(f"BERTScore unavailable — skipping: {error!r}")
        print("Run the setup cell and RESTART the session to enable it.")
        return
    updates = {}
    for (platform, direction, this_size), df in sorted(results.items()):
        if this_size != size:
            continue
        lang = direction.split("-")[1]  # language of translation and gold
        _, _, f1 = bert_score(df["translation"], df["gold"], lang=lang)
        mean_f1 = f1.mean().item()
        print(f"{platform} {direction} {this_size}: BERTScore F1 = {mean_f1:.4f}")
        updates[(platform, direction, this_size)] = {
            "BERTScore": f"{mean_f1:.4f}",
            "sentences": len(df["text"]),
        }
    if updates:
        save_scores(updates)


def run_all(size):
    evaluate(size)
    print()
    comet_evaluate(size)
    print()
    bertscore_evaluate(size)


In [8]:
# @title
# ===== Run ALL metrics on SHORT data (100 sentences per direction) =====
# chrF/BLEU + bootstrap + disagreements + COMET + BERTScore, all saved
# into evaluation/. Fast smoke test; COMET/BERTScore are skipped with a
# message if the setup cell hasn't been run.
run_all("short")


===== SHORT DATA =====

platform  direction    sent    chrF    BLEU
android   en-pl         100   60.88   34.73
ios       en-pl         100   70.68   46.90
android   pl-en         100   62.62   42.50
ios       pl-en         100   74.36   59.77

en-pl paired bootstrap (n=100, sample=100, 1000 iterations): iOS wins 100.0%, Android wins 0.0% -> SIGNIFICANT
Wrote evaluation/disagreements_en-pl_short.csv

en-pl: 10 largest disagreements
src:              It's my brother's.
ios (100.0):      To jest mojego brata.
android ( 17.0):  To mój brat.
gold:             To jest mojego brata.

src:              He's an Englishman.
ios (100.0):      On jest Anglikiem.
android ( 18.7):  Jest angielski.
gold:             On jest Anglikiem.

src:              He is English.
ios (100.0):      On jest Anglikiem.
android ( 20.6):  Jest angielskim.
gold:             On jest Anglikiem.

src:              He made his parents happy.
ios ( 80.3):      Uszczęśliwił swoich rodziców.
android ( 32.3):  Zrobił rodzice

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


android pl-en short: BERTScore F1 = 0.9627
ios en-pl short: BERTScore F1 = 0.9291


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ios pl-en short: BERTScore F1 = 0.9772
Updated evaluation/scores.csv


In [9]:
# ===== Run ALL metrics on FULL data (~78k / ~75k per direction) =====
# The thesis run. Use a GPU runtime (Colab T4) — COMET and BERTScore
# detect it automatically. Expect roughly an hour-plus for the neural
# metrics; chrF/BLEU + bootstrap take a few minutes on CPU.
run_all("full")


===== FULL DATA =====

platform  direction    sent    chrF    BLEU
android   en-pl       75390   62.49   38.83
ios       en-pl       75390   72.92   53.22
android   pl-en       78296   61.62   40.57
ios       pl-en       78296   73.22   60.38

en-pl paired bootstrap (n=75390, sample=10000, 1000 iterations): iOS wins 100.0%, Android wins 0.0% -> SIGNIFICANT
Wrote evaluation/disagreements_en-pl_full.csv

en-pl: 10 largest disagreements
src:              Endure!
ios (100.0):      Wytrzymaj!
android (  1.8):  Znosić!
gold:             Wytrzymaj!

src:              Squeeze.
ios (100.0):      Ściśnij.
android (  2.1):  Squeeze.
gold:             Ściśnij.

src:              Run.
ios (100.0):      Biegnij.
android (  2.1):  Uruchom.
gold:             Biegnij.

src:              Suck my dick!
ios (100.0):      Ssij mojego kutasa!
android (  2.1):  SSSE MY DICK!
gold:             Ssij mojego kutasa!

src:              Nonsense!
ios (  2.1):      Bzdura!
android (100.0):  Nonsens!
gold:          

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experim

android en-pl full: COMET = 0.8949


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 2447/2447 [13:25<00:00,  3.04it/s]
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.

...

ios pl-en full: COMET = 0.9264
Updated evaluation/scores.csv

android en-pl full: BERTScore F1 = 0.9059


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


android pl-en full: BERTScore F1 = 0.9619
ios en-pl full: BERTScore F1 = 0.9338


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ios pl-en full: BERTScore F1 = 0.9781
Updated evaluation/scores.csv


In [10]:
# ===== Download evaluation outputs =====
# Gathers everything the evaluation flows wrote into evaluation/
# (scores.csv, disagreements_*.csv, ...) into one zip archive.
# On Colab the browser download starts automatically; locally the zip
# just lands next to the notebook.
import shutil
from pathlib import Path

if not Path("evaluation").is_dir():
    print("No evaluation/ folder yet — run the evaluation flows first.")
else:
    archive = shutil.make_archive(
        "translation-evaluation", "zip", root_dir=".", base_dir="evaluation"
    )
    print(f"Created {archive}, containing:")
    for path in sorted(Path("evaluation").iterdir()):
        print(f"  {path.name} ({path.stat().st_size:,} bytes)")

    try:
        from google.colab import files
    except ImportError:
        print("Not running on Colab — download skipped, the zip is next to the notebook.")
    else:
        files.download(archive)


Created /content/TranslatorData/translation-evaluation.zip, containing:
  disagreements_en-pl_full.csv (596 bytes)
  disagreements_en-pl_short.csv (1,252 bytes)
  disagreements_pl-en_full.csv (865 bytes)
  disagreements_pl-en_short.csv (1,174 bytes)
  full-eval-output.txt (3,361 bytes)
  scores.csv (519 bytes)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>